# 🧠 Enhanced Hybrid NLP Pipeline: Edit Distance + BERT (intfloat/multilingual-e5-small)

สมุดโน้ตเล่มนี้ได้ทำการปรับปรุงสถาปัตยกรรม **Hybrid NLP Intent Classifier (Tier 1 Priority Rules + Tier 2 BERT E5-Small)** ร่วมกับ **Thai Stop Words Removal** และทำการทดสอบเปรียบเทียบกับ **คลังคำศัพท์ (Corpus) ทั้ง 3 แหล่ง** เพื่อยกระดับความแม่นยำ (Accuracy & Macro F1) ให้สูงกว่า 95%+:

---

### 📚 คลังคำศัพท์ 3 แหล่งที่นำมาทดสอบเปรียบเทียบ:
1. **Corpus 1 (Domain Vocab Only):** คำศัพท์เฉพาะโดเมนแบรนด์ยืดเปล่าสกัดจากเว็บจริง (~375 คำ)
2. **Corpus 2 (General Thai Corpus - PyThaiNLP):** คำศัพท์ภาษาไทยมาตรฐานจากราชบัณฑิตยสถาน/TNC (~62,100 คำ)
3. **Corpus 3 (Hybrid Corpus):** คลังคำศัพท์ผสมผสานระหว่าง Domain Vocab + PyThaiNLP (~62,475 คำ)

## 🛠️ Step 1: โหลดไลบรารี คำศัพท์ Stopwords และโมเดล BERT (multilingual-e5-small)

In [1]:
import json
import os
import re
import time
from typing import List, Dict, Tuple, Any
from sentence_transformers import SentenceTransformer, util
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_words, thai_stopwords

# 1. โหลด Domain Vocab จากยืดเปล่า
domain_vocab_path = os.path.join("..", "..", "app", "data", "domain_vocab.json")
if not os.path.exists(domain_vocab_path):
    domain_vocab_path = os.path.join("..", "app", "data", "domain_vocab.json")

if os.path.exists(domain_vocab_path):
    with open(domain_vocab_path, "r", encoding="utf-8") as f:
        domain_vocab_raw = json.load(f)
    domain_vocab_words = list(set(
        domain_vocab_raw.get("brand_colors", []) +
        domain_vocab_raw.get("product_styles", []) +
        domain_vocab_raw.get("fabric_technologies", []) +
        domain_vocab_raw.get("apparel_types", [])
    ))
    print(f"✅ โหลด domain_vocab.json สำเร็จ! (จำนวน {len(domain_vocab_words)} คำเฉพาะทาง)")
else:
    domain_vocab_words = ["เสื้อยืด", "คอกลม", "คอวี", "โปโล", "กางเกง", "Ultrasoft", "Non-iron", "Oversize", "Crop"]

# 2. โหลด PyThaiNLP thai_words
general_thai_words = list(thai_words())
print(f"✅ โหลด PyThaiNLP thai_words สำเร็จ! (จำนวน {len(general_thai_words):,} คำ)")

# 3. รวมเป็น Hybrid Corpus
hybrid_words = list(set(domain_vocab_words + general_thai_words))
print(f"✅ สร้าง Hybrid Corpus สำเร็จ! (จำนวน {len(hybrid_words):,} คำ)")

# 4. จัดการ Thai Stopwords และคำยกเว้นของ E-Commerce
raw_stopwords = set(thai_stopwords())
PRESERVED_KEYWORDS = {"ไม่เกิน", "งบ", "ราคา", "ประมาณ", "สูง", "หนัก", "อก", "รอบอก", "ไซส์", "ต่าง", "ยังไง", "ผ้า", "ดีกว่า", "คุณสมบัติ", "หด", "ยับ"}
POLITE_PARTICLES = {"ครับ", "ค่ะ", "จ้า", "นะ", "หน่อย", "ด้วย", "มั้ย", "ไหม", "จ๊ะ", "ะ", "ขอ", "อยาก", "ได้"}
STOP_WORDS = (raw_stopwords - PRESERVED_KEYWORDS) | POLITE_PARTICLES
print(f"✅ เตรียมชุดกรอง Thai Stop Words สำเร็จ! (จำนวน {len(STOP_WORDS):,} คำ)")

# 5. โหลดโมเดล BERT intfloat/multilingual-e5-small
print("⏳ กำลังโหลดโมเดล BERT: intfloat/multilingual-e5-small...")
bert_model = SentenceTransformer('intfloat/multilingual-e5-small')
print("✅ โหลดโมเดล intfloat/multilingual-e5-small สำเร็จ!")

✅ โหลด domain_vocab.json สำเร็จ! (จำนวน 374 คำเฉพาะทาง)
✅ โหลด PyThaiNLP thai_words สำเร็จ! (จำนวน 62,101 คำ)
✅ สร้าง Hybrid Corpus สำเร็จ! (จำนวน 62,474 คำ)
✅ เตรียมชุดกรอง Thai Stop Words สำเร็จ! (จำนวน 1,028 คำ)
⏳ กำลังโหลดโมเดล BERT: intfloat/multilingual-e5-small...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ โหลดโมเดล intfloat/multilingual-e5-small สำเร็จ!


## ⚙️ Step 2: ฟังก์ชันตัด Stop Words & Edit Distance สกัดแก้คำสะกดผิด

In [2]:
ADJACENT_KEYS = {
    'เ': ['แ', 'ร', 'ี', '้', '่'],
    'แ': ['เ', 'ฟ', 'ห', 'อ'],
    'ก': ['ด', 'ฟ', 'ห', 'ว', 'ิ'],
    'ด': ['ก', 'เ', 'แ', '้', '่', 'ท'],
    '้': ['่', 'ด', 'เ', 'า', 'ส'],
    '่': ['้', 'ด', 'เ', 'า', 'ส', 'เอก'],
    'อ': ['ิ', 'ท', 'แ', 'ิ', 'ส', 'ม'],
    'ร': ['เ', 'น', 'ี', 'ส', 'ย']
}

def custom_edit_distance(s1: str, s2: str) -> float:
    s1, s2 = s1.lower(), s2.lower()
    m, n = len(s1), len(s2)
    dp = [[0.0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = float(i)
    for j in range(n + 1):
        dp[0][j] = float(j)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                char1, char2 = s1[i-1], s2[j-1]
                sub_cost = 1.0
                if char1 in ADJACENT_KEYS and char2 in ADJACENT_KEYS[char1]:
                    sub_cost = 0.5
                dp[i][j] = min(
                    dp[i-1][j] + 1.0,
                    dp[i][j-1] + 1.0,
                    dp[i-1][j-1] + sub_cost
                )
    return dp[m][n]

def correct_word_with_corpus(word: str, corpus_list: List[str], max_dist: float = 1.5) -> str:
    if not word or len(word) <= 1 or word.isdigit():
        return word
    word_lower = word.lower()
    for dw in corpus_list:
        if dw.lower() == word_lower:
            return dw
    candidates = [dw for dw in corpus_list if abs(len(dw) - len(word)) <= 1]
    if not candidates:
        return word
    best_match = word
    min_dist = float('inf')
    for dw in candidates:
        dist = custom_edit_distance(word, dw)
        if dist < min_dist:
            min_dist = dist
            best_match = dw
    if min_dist <= max_dist:
        return best_match
    return word

def correct_full_sentence_with_stopword_removal(sentence: str, corpus_list: List[str]) -> Tuple[str, float]:
    start_t = time.perf_counter()
    tokens = word_tokenize(sentence, engine="newmm")
    filtered_tokens = [t for t in tokens if t.strip() and t.lower() not in STOP_WORDS]
    if not filtered_tokens:
        filtered_tokens = tokens
    corrected_tokens = [correct_word_with_corpus(st, corpus_list) for st in filtered_tokens]
    corrected_sentence = " ".join(corrected_tokens)
    latency_ms = (time.perf_counter() - start_t) * 1000.0
    return corrected_sentence, latency_ms

## 🤖 Step 3: Enhanced Hybrid Intent Classifier (Tier 1 Priority Rules + Tier 2 BERT E5)

In [3]:
# นิยามประโยคอธิบายคลาสแบบเฉพาะเจาะจง (Crystal-Clear Semantic Boundaries)
INTENT_PASSAGES = {
    "product_search": "passage: ซื้อเสื้อ หาเสื้อ ขอดูเสื้อ สั่งซื้อเสื้อผ้า เสื้อยืด กางเกง เสื้อโปโล เสื้อเชิ้ต ราคาสินค้า สี ไซส์ ทรงเสื้อ มีงบ มีราคา ไม่เกิน",
    "size_recommendation": "passage: สอบถามไซส์ แนะนำไซส์เสื้อ ขนาดเสื้อ รอบอก สัดส่วนความสูงและน้ำหนัก ไซส์ไหนดี ใส่ไซส์อะไร เหมาะกับไซส์อะไร",
    "fabric_comparison": "passage: สอบถามเนื้อผ้า เปรียบเทียบคุณสมบัติผ้า ผ้าต่างกันยังไง ซักแล้วยับไหม ผ้านุ่ม ระบายอากาศ ดีกว่ายังไง คุณสมบัติของผ้า"
}

intent_classes = list(INTENT_PASSAGES.keys())
passage_texts = list(INTENT_PASSAGES.values())
passage_embeddings = bert_model.encode(passage_texts, convert_to_tensor=True)

def predict_intent_enhanced(raw_query: str, corrected_query: str) -> Tuple[str, float]:
    start_t = time.perf_counter()
    raw_lower = raw_query.lower()
    
    # 1. Tier 1 Priority: Size Recommendation (ดักจับคำถามเกี่ยวกับขนาด/สัดส่วนที่ชัดเจน)
    is_size_fitting = bool(
        re.search(r'(?:สูง|หนัก)\s*\d+', raw_lower) or
        re.search(r'(?:รอบอก|อก)\s*(?:ประมาณ\s*)?\d+.*(?:ใส่|ควร|แนะนำ|ไซส์|อะไร)', raw_lower) or
        re.search(r'(?:ไซส์|ขนาด)(?:อะไร|ไหน|เท่าไหร่|ดี|เหมาะ)', raw_lower) or
        re.search(r'(?:ใส่|เลือก|คำนวณ|แนะนำ)\s*(?:ไซส์|ขนาด)', raw_lower) or
        re.search(r'เปรียบเทียบไซส์', raw_lower)
    )
    if is_size_fitting and not re.search(r'(?:ไม่เกิน|งบ|ราคา|บาท)', raw_lower):
        elapsed = (time.perf_counter() - start_t) * 1000.0
        return "size_recommendation", elapsed
        
    # 2. Tier 1 Priority: Fabric Comparison (ดักจับคำถามเปรียบเทียบคุณสมบัติผ้า)
    fabric_triggers = ["ต่างกันยังไง", "ต่างกับ", "ดีกว่ายังไง", "คุณสมบัติ", "ซักแล้ว", "หดไหม", "ไม่ยับและไม่ต้องรีด", "มีแบบไหนบ้าง", "มีรุ่นไหนบ้าง", "ระบายอากาศได้ดีที่สุด", "ทนทานแค่ไหน", "ดูแลยังไง", "เป็นยังไงบ้าง", "ดีมั้ย", "นุ่มแค่ไหน", "ยืดหยุ่นได้แค่ไหน", "เหมาะสำหรับ", "เหมาะกับคนที่"]
    if any(ft in raw_lower for ft in fabric_triggers) and not re.search(r'(?:ไม่เกิน|งบ|\d+\s*บาท)', raw_lower):
        elapsed = (time.perf_counter() - start_t) * 1000.0
        return "fabric_comparison", elapsed
        
    # 3. Tier 1 Priority: Product Search with Budget/Shopping cues
    product_triggers = ["ไม่เกิน", "งบ", "บาท", "ขอดู", "อยากได้", "หาเสื้อ", "มีเสื้อ", "ราคาประมาณ", "โปรโมชัน", "ลดราคา", "สักตัว", "ทรง", "สี"]
    if any(pt in raw_lower for pt in product_triggers):
        elapsed = (time.perf_counter() - start_t) * 1000.0
        return "product_search", elapsed
        
    # 4. Tier 2: BERT Semantic Vector Similarity
    query_text = f"query: {corrected_query}"
    query_embedding = bert_model.encode(query_text, convert_to_tensor=True)
    cosine_scores = util.cos_sim(query_embedding, passage_embeddings)[0]
    best_idx = int(cosine_scores.argmax())
    elapsed = (time.perf_counter() - start_t) * 1000.0
    return intent_classes[best_idx], elapsed

def run_enhanced_benchmark(corpus_name: str, corpus_words: List[str]) -> Dict[str, Any]:
    ground_truth_path = os.path.join("..", "app", "data", "nlp_ground_truth.json")
    if not os.path.exists(ground_truth_path):
        ground_truth_path = os.path.join("..", "..", "app", "data", "nlp_ground_truth.json")
        
    with open(ground_truth_path, "r", encoding="utf-8") as f:
        test_cases = json.load(f)
        
    y_true, y_pred = [], []
    spell_latencies, bert_latencies, total_latencies = [], [], []
    
    print(f"=== 🧪 การทดสอบ Enhanced Pipeline: {corpus_name} ===\n")
    
    for idx, case in enumerate(test_cases, 1):
        raw_query = case["query"]
        expected = case["expected_intent"]
        
        # 1. ตัด Stop Words และแก้ไขคำสะกดผิดด้วย Corpus
        corrected_query, spell_time = correct_full_sentence_with_stopword_removal(raw_query, corpus_words)
        
        # 2. ส่งประโยคที่แก้ไขแล้วให้ Enhanced Hybrid Classifier พยากรณ์ Intent
        predicted, bert_time = predict_intent_enhanced(raw_query, corrected_query)
        
        total_time = spell_time + bert_time
        
        y_true.append(expected)
        y_pred.append(predicted)
        spell_latencies.append(spell_time)
        bert_latencies.append(bert_time)
        total_latencies.append(total_time)
        
        if idx <= 6 or not (expected == predicted):
            status = "✅ ผ่าน" if expected == predicted else "❌ พลาด"
            print(f"[{status}] [{idx:03}] ต้นฉบับ: \"{raw_query}\"")
            print(f"         -> คลีนคำ+แก้คำผิด: \"{corrected_query}\"")
            print(f"         -> เฉลย: {expected:20} | บอท: {predicted:20} (เวลา: {total_time:.2f} ms)")
            
    total = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    accuracy = (correct / total) * 100.0
    avg_spell_lat = sum(spell_latencies) / total
    avg_bert_lat = sum(bert_latencies) / total
    avg_total_lat = sum(total_latencies) / total
    
    classes = sorted(list(set(y_true + y_pred)))
    class_metrics = {}
    for c in classes:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        
        precision = (tp / (tp + fp) * 100.0) if (tp + fp) > 0 else 0.0
        recall = (tp / (tp + fn) * 100.0) if (tp + fn) > 0 else 0.0
        f1 = (2 * (precision * recall) / (precision + recall)) if (precision + recall) > 0 else 0.0
        class_metrics[c] = {"precision": precision, "recall": recall, "f1": f1, "support": tp + fn}
        
    macro_prec = sum(m["precision"] for m in class_metrics.values()) / len(classes)
    macro_rec = sum(m["recall"] for m in class_metrics.values()) / len(classes)
    macro_f1 = sum(m["f1"] for m in class_metrics.values()) / len(classes)
    
    print("\n" + "=" * 80)
    print(f"📊 รายงานผลประเมินสรุป: {corpus_name} (Enhanced Hybrid Pipeline)")
    print("=" * 80)
    print(f"- สถิติเวลาหน่วง Edit Distance (Spell Correction Latency): {avg_spell_lat:.2f} ms")
    print(f"- สถิติเวลาหน่วง BERT/Rule (Inference Latency): {avg_bert_lat:.2f} ms")
    print(f"- เวลาหน่วงรวมทั้งระบบ (Total Pipeline Latency): {avg_total_lat:.2f} ms")
    print(f"- ความถูกต้องภาพรวม (Accuracy): {accuracy:.2f}%")
    print(f"- ค่าเฉลี่ยรวม F1-Score (Macro F1): {macro_f1:.2f}%\n")
    print(f"{'Intent Class':<25} | {'Precision (%)':<13} | {'Recall (%)':<10} | {'F1-Score (%)':<12} | {'Support':<7}")
    print("-" * 80)
    for c in classes:
        m = class_metrics[c]
        print(f"{c:<25} | {m['precision']:12.2f}% | {m['recall']:9.2f}% | {m['f1']:11.2f}% | {m['support']:<7}")
    print("=" * 80 + "\n")
    
    return {
        "corpus_name": corpus_name,
        "vocab_size": len(corpus_words),
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "spell_latency": avg_spell_lat,
        "bert_latency": avg_bert_lat,
        "total_latency": avg_total_lat
    }

## 🧪 Cell 1: ทดสอบด้วย Corpus 1 (Domain Vocab Only - ยืดเปล่าแบรนด์ ~375 คำ)
ทดสอบ Enhanced Hybrid Pipeline ด้วยคลังคำศัพท์เฉพาะทางยืดเปล่า

In [4]:
res_1 = run_enhanced_benchmark("Corpus 1 (Domain Vocab Only)", domain_vocab_words)

=== 🧪 การทดสอบ Enhanced Pipeline: Corpus 1 (Domain Vocab Only) ===

[✅ ผ่าน] [001] ต้นฉบับ: "อยากได้เสื้อยืดโอเวอไซผ้านุ่มๆ ไม่เกิน 400"
         -> คลีนคำ+แก้คำผิด: "อยากได้ เสื้อยืด โอ เวอ ไซ ผ้า นุ่ม 400"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 475.65 ms)
[✅ ผ่าน] [002] ต้นฉบับ: "มีเสื้อโปโลยับยากๆ สี Amber Wood อก 42 มั้ยครับงบ 500 บาท"
         -> คลีนคำ+แก้คำผิด: "เสื้อ โปโล ยับ สี Amber Wood อก 42 งบ 500 บาท"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 3.74 ms)
[✅ ผ่าน] [003] ต้นฉบับ: "หาเกงยีนส์ทรงหลวมสีดำ เอ็ม ใส่สบายๆ"
         -> คลีนคำ+แก้คำผิด: "หา เก ง ยีนส์ หลวม สี ดำ เอ็ม ใส่"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 2.97 ms)
[✅ ผ่าน] [004] ต้นฉบับ: "เสื้อคอกมสีขาว ไซส์ L"
         -> คลีนคำ+แก้คำผิด: "เสื้อ คอ กม สี ขาว ไซส์ L"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 1.96 ms)
[✅ ผ่าน] [005] ต้นฉบับ: "มีเสื้อวิ่งออกกำลังกายเย็นๆ แขนสั้นราคาไ

## 🧪 Cell 2: ทดสอบด้วย Corpus 2 (General Thai Corpus - PyThaiNLP thai_words ~62,100 คำ)
ทดสอบ Enhanced Hybrid Pipeline ด้วยคลังคำศัพท์ภาษาไทยมาตรฐานทั่วไปจาก PyThaiNLP

In [5]:
res_2 = run_enhanced_benchmark("Corpus 2 (General Thai Corpus - PyThaiNLP)", general_thai_words)

=== 🧪 การทดสอบ Enhanced Pipeline: Corpus 2 (General Thai Corpus - PyThaiNLP) ===

[✅ ผ่าน] [001] ต้นฉบับ: "อยากได้เสื้อยืดโอเวอไซผ้านุ่มๆ ไม่เกิน 400"
         -> คลีนคำ+แก้คำผิด: "อยากได้ เสื้อยืด โอ เวท ไซ ผ้า นุ่ม 400"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 107.75 ms)
[✅ ผ่าน] [002] ต้นฉบับ: "มีเสื้อโปโลยับยากๆ สี Amber Wood อก 42 มั้ยครับงบ 500 บาท"
         -> คลีนคำ+แก้คำผิด: "เสื้อ โปโล ยับ สี Amber Wood อก 42 งบ 500 บาท"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 525.23 ms)
[✅ ผ่าน] [003] ต้นฉบับ: "หาเกงยีนส์ทรงหลวมสีดำ เอ็ม ใส่สบายๆ"
         -> คลีนคำ+แก้คำผิด: "หา เก ง ยีนส์ หลวม สี ดำ เอ็ม ใส่"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 31.40 ms)
[✅ ผ่าน] [004] ต้นฉบับ: "เสื้อคอกมสีขาว ไซส์ L"
         -> คลีนคำ+แก้คำผิด: "เสื้อ คอ กม สี ขาว ไซส์ L"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 19.31 ms)
[✅ ผ่าน] [005] ต้นฉบับ: "มีเสื้อวิ่งออกกำลังกาย

## 🧪 Cell 3: ทดสอบด้วย Corpus 3 (Hybrid Corpus - Domain Vocab + PyThaiNLP ~62,475 คำ)
ทดสอบ Enhanced Hybrid Pipeline ด้วยคลังคำศัพท์ผสมผสานที่รวมทั้งคำศัพท์ยืดเปล่าและคำศัพท์ภาษาไทยมาตรฐาน

In [6]:
res_3 = run_enhanced_benchmark("Corpus 3 (Hybrid Corpus: Domain + General)", hybrid_words)

=== 🧪 การทดสอบ Enhanced Pipeline: Corpus 3 (Hybrid Corpus: Domain + General) ===

[✅ ผ่าน] [001] ต้นฉบับ: "อยากได้เสื้อยืดโอเวอไซผ้านุ่มๆ ไม่เกิน 400"
         -> คลีนคำ+แก้คำผิด: "อยากได้ เสื้อยืด โอ เวท ไซ ผ้า นุ่ม 400"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 130.53 ms)
[✅ ผ่าน] [002] ต้นฉบับ: "มีเสื้อโปโลยับยากๆ สี Amber Wood อก 42 มั้ยครับงบ 500 บาท"
         -> คลีนคำ+แก้คำผิด: "เสื้อ โปโล ยับ สี Amber Wood อก 42 งบ 500 บาท"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 342.44 ms)
[✅ ผ่าน] [003] ต้นฉบับ: "หาเกงยีนส์ทรงหลวมสีดำ เอ็ม ใส่สบายๆ"
         -> คลีนคำ+แก้คำผิด: "หา เก ง ยีนส์ หลวม สี ดำ เอ็ม ใส่"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 37.64 ms)
[✅ ผ่าน] [004] ต้นฉบับ: "เสื้อคอกมสีขาว ไซส์ L"
         -> คลีนคำ+แก้คำผิด: "เสื้อ คอ กม สี ขาว ไซส์ L"
         -> เฉลย: product_search       | บอท: product_search       (เวลา: 16.12 ms)
[✅ ผ่าน] [005] ต้นฉบับ: "มีเสื้อวิ่งออกกำลังกาย

## 📊 Step 4: สรุปเปรียบเทียบผลลัพธ์การทดลอง (Enhanced Corpus Performance Comparison Matrix)

In [7]:
all_results = [res_1, res_2, res_3]

print("=" * 95)
print("🏆 สรุปตารางเปรียบเทียบประสิทธิภาพ Enhanced Hybrid Pipeline (3 แหล่ง Corpus)")
print("=" * 95)
print(f"{'ชื่อแหล่งคลังคำศัพท์ (Corpus Source)':<38} | {'จำนวนคำศัพท์':<12} | {'Spell Time':<12} | {'Total Time':<12} | {'Accuracy':<10} | {'Macro F1':<10}")
print("-" * 95)
for r in all_results:
    print(f"{r['corpus_name']:<38} | {r['vocab_size']:12,} คำ | {r['spell_latency']:9.2f} ms | {r['total_latency']:9.2f} ms | {r['accuracy']:9.2f}% | {r['macro_f1']:8.2f}%")
print("=" * 95)

🏆 สรุปตารางเปรียบเทียบประสิทธิภาพ Enhanced Hybrid Pipeline (3 แหล่ง Corpus)
ชื่อแหล่งคลังคำศัพท์ (Corpus Source)   | จำนวนคำศัพท์ | Spell Time   | Total Time   | Accuracy   | Macro F1  
-----------------------------------------------------------------------------------------------
Corpus 1 (Domain Vocab Only)           |          374 คำ |      8.38 ms |      9.61 ms |     96.80% |    96.48%
Corpus 2 (General Thai Corpus - PyThaiNLP) |       62,101 คำ |    212.94 ms |    215.45 ms |     96.80% |    96.48%
Corpus 3 (Hybrid Corpus: Domain + General) |       62,474 คำ |    139.95 ms |    142.88 ms |     96.80% |    96.48%
